# Experiment 15 — Evre 2 / 3 / 4: RGB ve RGB + Doku — Colab

**Amaç:** Evre 1 eğitimden çıkarılınca tahmini doku haritaları, Evre 2–3–4 ayrımını iyileştiriyor mu?

Aynı görüntüler ve aynı fold'lar üzerinde iki yeni model eğitilir:

| Model | Girdi | Çıktı |
|---|---|---|
| `rgb` | RGB | Evre 2 / 3 / 4 |
| `rgb_tissue` | RGB + 5 tahmini doku kanalı, late fusion / concat | Evre 2 / 3 / 4 |

İkisi de **ImageNet MiT-B1 + CORN** ile başlar. Önceden eğitilmiş dört sınıflı evreleme modelleri bu karşılaştırmada kullanılmaz. Önce orijinal dört sınıflı fold'lar oluşturulur; ardından her eğitim/doğrulama/test listesinden Evre 1 çıkarılır ve 2/3/4 etiketleri 0/1/2'ye çevrilir. CORN başlığı **iki eşik çıktısı** üretir.

**Gerekli veri:** Önceki exp-3 / exp-4 deneylerindeki `wound-staging-masks-exp-3.zip`. Yeni maske veya doku segmentasyonu üretmeye gerek yoktur. Veri arşivinde dört evrenin de bulunması gerekir; Evre 1 yalnızca orijinal fold'ları yeniden kurmak için okunur.

**Çalıştırma:** Colab'da **Runtime → Change runtime type → GPU** seçin ve hücreleri sırayla çalıştırın. Tam deney **2 model × 3 fold = 6 eğitim**, varsayılan **100 epoch** sürer. Kesinti olursa aynı defteri ve aynı çıktı klasörünü kullanarak devam edin. Son hücre sonuçları ZIP olarak indirir; büyük model dosyaları Drive'da kalır.

Bu indirilebilir defter deney kodunu içinde taşır; GitHub'a yeni kod gönderilmiş olması gerekmez.

## 1. Drive'ı bağla

Daha önce kullandığınız `wound-staging-masks-exp-3.zip` dosyası varsayılan olarak `MyDrive/thesis/` altında aranır.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Veri ve çıktı konumları

Aynı deneyi sürdürürken `OUTPUT_DIR` değerini değiştirmeyin. Farklı epoch veya batch ayarıyla yeni bir deney başlatacaksanız yeni bir çıktı klasörü kullanın. Hızlı kontrolün çıktıları ayrı bir klasöre yazılır.

In [ ]:
from pathlib import Path

REPO_DIR = Path('/content/CanRollasThesisProject')
DRIVE_THESIS_DIR = Path('/content/drive/MyDrive/thesis')
ZIP_PATH = DRIVE_THESIS_DIR / 'wound-staging-masks-exp-3.zip'
EXTRACT_DIR = Path('/content/data/exp15_stage234_tissue')
OUTPUT_DIR = DRIVE_THESIS_DIR / 'exp15_stage234_tissue_outputs'

# Veri zaten Colab'a açıldıysa yolunu yazabilirsiniz; boş bırakılırsa bulunur/açılır.
DATA_DIR_OVERRIDE = ''

# Ana karşılaştırma: iki model, üç fold. Varsayılan protokolü koruyun.
VARIANTS = 'rgb,rgb_tissue'
FOLDS = '0,1,2'
WORKERS = 2
BATCH_SIZE = 16
EPOCHS = None  # None = configs/base.yaml içindeki 100 epoch

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Veri ZIP:', ZIP_PATH)
print('Kalıcı çıktı klasörü:', OUTPUT_DIR)

## 3. Deney kodunu hazırla

Deney kodu ve gereken ortak modüller bu defterin içinde bulunur. Hücre bunları Colab'a açar; GitHub güncellemesi veya ayrı kaynak dosyası yüklemeniz gerekmez. Bu hücrenin kodunu değiştirmeniz gerekmez.

In [ ]:
#@title Deney kodunu hazırla
import base64
import io
import subprocess
import zipfile

BUNDLED_SOURCE_ZIP_BASE64 = "..."  # Generated by scripts/bundle_notebook.py

if BUNDLED_SOURCE_ZIP_BASE64:
    REPO_DIR.mkdir(parents=True, exist_ok=True)
elif not REPO_DIR.exists():
    clone = subprocess.run(
        ['git', 'clone', 'https://github.com/canrollas/CanRollasThesisProject.git', str(REPO_DIR)],
        check=False,
    )
    if clone.returncode and not BUNDLED_SOURCE_ZIP_BASE64:
        raise RuntimeError('Repo indirilemedi. Gömülü kod içeren güncel defteri kullanın.')
    REPO_DIR.mkdir(parents=True, exist_ok=True)
elif not BUNDLED_SOURCE_ZIP_BASE64 and (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

if BUNDLED_SOURCE_ZIP_BASE64:
    source_bytes = base64.b64decode(BUNDLED_SOURCE_ZIP_BASE64)
    with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
        for member in archive.infolist():
            target = (REPO_DIR / member.filename).resolve()
            if not target.is_relative_to(REPO_DIR.resolve()):
                raise ValueError(f'Geçersiz arşiv yolu: {member.filename}')
        archive.extractall(REPO_DIR)
    print('Deftere gömülü deney kodu hazırlandı.')

EXPERIMENT_DIR = REPO_DIR / 'more_experiments/exp/exp15_stage234_tissue'
TRAIN_SCRIPT = EXPERIMENT_DIR / 'scripts/train.py'
assert TRAIN_SCRIPT.is_file(), 'train.py bulunamadı; gömülü kod içeren güncel defteri kullanın.'
print('Deney klasörü:', EXPERIMENT_DIR)

## 4. Veriyi bul / ZIP'ten çıkar ve kontrol et

İç içe arşiv klasörleri otomatik bulunur. ZIP dosyası kullanılırken eğitim verileri Colab'ın yerel diskine açılır; Drive üzerinden her batch'te görüntü okumaya gerek kalmaz.

In [ ]:
import numpy as np

DATASET_NAME = 'wound-staging-masks-exp-3'
STAGES = ['stage_1', 'stage_2', 'stage_3', 'stage_4']
EXPECTED_COUNTS = {'stage_1': 230, 'stage_2': 313, 'stage_3': 275, 'stage_4': 273}


def is_dataset_root(path):
    path = Path(path)
    return all((path / stage).is_dir() and len(list((path / stage).glob('*.npy'))) == EXPECTED_COUNTS[stage] for stage in STAGES)


def find_dataset_root():
    candidates = [REPO_DIR / 'datasets' / DATASET_NAME, EXTRACT_DIR / DATASET_NAME, EXTRACT_DIR]
    if EXTRACT_DIR.exists():
        candidates.extend(sorted(EXTRACT_DIR.rglob(DATASET_NAME)))
        candidates.extend(p.parent for p in sorted(EXTRACT_DIR.rglob('stage_1')) if p.is_dir())
    return next((p.resolve() for p in candidates if is_dataset_root(p)), None)


if DATA_DIR_OVERRIDE:
    DATA_ROOT = Path(DATA_DIR_OVERRIDE).expanduser().resolve()
    assert is_dataset_root(DATA_ROOT), f'Dört evreyi içeren .npy veri klasörü bulunamadı: {DATA_ROOT}'
else:
    DATA_ROOT = find_dataset_root()
    if DATA_ROOT is None:
        assert ZIP_PATH.is_file(), f'ZIP bulunamadı: {ZIP_PATH}. Dosyayı MyDrive/thesis klasörüne koyun.'
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH) as archive:
            for member in archive.infolist():
                if not (EXTRACT_DIR / member.filename).resolve().is_relative_to(EXTRACT_DIR.resolve()):
                    raise ValueError(f'Geçersiz arşiv yolu: {member.filename}')
            archive.extractall(EXTRACT_DIR)
        DATA_ROOT = find_dataset_root()
    assert DATA_ROOT is not None, f'Arşivde {DATASET_NAME} verisi bulunamadı.'

counts = {}
for stage in STAGES:
    paths = sorted((DATA_ROOT / stage).glob('*.npy'))
    counts[stage] = len(paths)
    sample = np.load(paths[0], mmap_mode='r', allow_pickle=False)
    assert sample.ndim == 3 and sample.shape[-1] == 8, f'{paths[0]}: (H,W,8) bekleniyor, gelen {sample.shape}'
    assert sample.dtype == np.uint8, f'{paths[0]}: uint8 bekleniyor, gelen {sample.dtype}'
assert counts == EXPECTED_COUNTS, f'Veri eksik veya farklı: {counts}; beklenen {EXPECTED_COUNTS}'
print('Veri klasörü:', DATA_ROOT)
print('Görüntü sayıları:', counts)
print('Üç sınıflı deneyde kullanılacak toplam görüntü:', sum(counts[s] for s in STAGES[1:]))

## 5. Bağımlılıklar ve GPU kontrolü

Task 3 ile aynı eğitim kütüphaneleri kullanılır. İlk çalıştırmada ImageNet ağırlıkları indirilir.

In [ ]:
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'task_3/requirements.txt'), 'pandas>=2.0'],
    check=True,
)

import torch
import yaml

assert torch.cuda.is_available(), 'GPU bulunamadı. Runtime → Change runtime type → GPU seçip hücreleri yeniden çalıştırın.'
print('GPU:', torch.cuda.get_device_name(0))
with (EXPERIMENT_DIR / 'configs/base.yaml').open() as handle:
    experiment_config = yaml.safe_load(handle)
print('Varsayılan eğitim ayarları:')
print(yaml.safe_dump(experiment_config, sort_keys=False))
subprocess.run([sys.executable, str(TRAIN_SCRIPT), '--help'], check=True, cwd=EXPERIMENT_DIR)

## 6. Kısa deneme (önerilir)

İki modeli çok küçük bir veri alt kümesinde iki epoch çalıştırır. Bu yalnızca kurulum ve eğitim akışını kontrol eder; çıkan skorlar deney sonucu değildir. Çıktıları ayrı `smoke/` alt klasöründe kalır ve tam eğitimin tamamlanmış fold'ları sayılmaz.

In [ ]:
RUN_SMOKE = True

if RUN_SMOKE:
    smoke_cmd = [
        sys.executable, '-u', str(TRAIN_SCRIPT),
        '--data-root', str(DATA_ROOT), '--output-dir', str(OUTPUT_DIR),
        '--variants', VARIANTS, '--folds', '0', '--workers', str(WORKERS), '--smoke',
    ]
    print('Kısa deneme çıktıları:', OUTPUT_DIR / 'smoke')
    subprocess.run(smoke_cmd, check=True, cwd=EXPERIMENT_DIR)
else:
    print('Kısa deneme atlandı.')

## 7. Tam eğitimi başlat / kaldığı yerden sürdür

Varsayılan: **100 epoch, batch 16, üç fold, iki model**. Her iki model aynı .npy dosyalarının RGB kanallarını kullanır; doku modelinde buna beş doku olasılık kanalı eklenir. Model seçimi yalnızca doğrulama macro-F1 ile yapılır; test bölümü son değerlendirme içindir.

Checkpoint'ler her epoch sonunda Drive'a kaydedilir. Colab kesilirse yeniden bağlanın, önceki hücreleri çalıştırıp bu hücreyi tekrar çalıştırın. Tamamlanan model/fold'lar atlanır, yarım kalan eğitim son kaydedilen epoch'tan sürer. Bir hata oluşursa hücre durur ve hatayı gösterir; düzeltildikten sonra tekrar çalıştırın.

In [ ]:
assert torch.cuda.is_available(), 'Tam eğitim için GPU runtime seçin.'
train_cmd = [
    sys.executable, '-u', str(TRAIN_SCRIPT),
    '--data-root', str(DATA_ROOT), '--output-dir', str(OUTPUT_DIR),
    '--variants', VARIANTS, '--folds', FOLDS, '--workers', str(WORKERS),
    '--batch', str(BATCH_SIZE),
]
if EPOCHS is not None:
    train_cmd.extend(['--epochs', str(EPOCHS)])
print('Tam eğitim çıktıları:', OUTPUT_DIR)
subprocess.run(train_cmd, check=True, cwd=EXPERIMENT_DIR)
print('Seçilen model/fold eğitimleri tamamlandı.')

## 8. Üç sınıflı sonuçları göster

Ana sonuç `rgb_tissue` ile bu deneyde yeniden eğitilen `rgb` arasındaki farktır. **Eski dört sınıflı RGB skoruyla doğrudan kıyaslamayın:** sınıflar ve değerlendirme kapsamı farklıdır. Macro-F1 yalnızca Evre 2, 3 ve 4 üzerinden hesaplanır. Fold ortalaması/standart sapması ile birleşik test sonuçları ayrı raporlanır; her fold'un farkı `paired_comparison.csv` içindedir.

In [ ]:
import pandas as pd
from IPython.display import display

RESULTS_DIR = OUTPUT_DIR / 'results'
results_path = RESULTS_DIR / 'results.csv'
assert results_path.is_file(), f'Sonuç dosyası bulunamadı: {results_path}'
results_df = pd.read_csv(results_path)
expected_runs = len(VARIANTS.split(',')) * len(FOLDS.split(','))
print(f'Tamamlanan eğitim: {len(results_df)}; bu seçimde beklenen: {expected_runs}')
display(results_df)
for filename in ['summary.csv', 'pooled_metrics.csv', 'paired_comparison.csv', 'per_class_metrics.csv', 'confusion_matrices.csv']:
    report_path = RESULTS_DIR / filename
    print('\n' + filename)
    if report_path.is_file():
        display(pd.read_csv(report_path))
    else:
        print('Henüz üretilmedi:', report_path)

## 9. İsteğe bağlı: Evre 1 kapısı ile tüm sistemi değerlendir

Bu adım ana üç sınıflı deneye ek bir değerlendirmedir; yeniden eğitim yapmaz. İlgili fold'un **önceden eğitilmiş dört sınıflı RGB modeli** Evre 1 derse o karar korunur; diğer görüntüler bu deneyde eğitilmiş üç sınıflı modele gider. Hem `rgb` hem `rgb_tissue` dalı aynı kapıyla karşılaştırılır; kapının hataları toplam dört sınıflı sonuca dahil edilir.

Varsayılan olarak kapalıdır. Çalıştırmak için `RUN_CASCADE = True` yapın ve `GATE_CHECKPOINTS_DIR` yolunu kontrol edin. Klasörde `mit_b1__corn_fold0.pt`, `mit_b1__corn_fold1.pt` ve `mit_b1__corn_fold2.pt` bulunmalıdır. Bunlar Task 3'ün aynı fold protokolüyle eğitilmiş RGB modelleri olmalıdır.

Bu adım orijinal Task 3 checkpoint’lerini ve exp-3 arşivindeki `metadata.csv` dosyasını doğrular. Farklı bir eğitimden gelen checkpoint’ler kabul edilmez.

In [ ]:
RUN_CASCADE = False
GATE_CHECKPOINTS_DIR = DRIVE_THESIS_DIR / 'task_3_outputs/checkpoints'

if RUN_CASCADE:
    for fold in [int(value) for value in FOLDS.split(',')]:
        checkpoint = GATE_CHECKPOINTS_DIR / f'mit_b1__corn_fold{fold}.pt'
        assert checkpoint.is_file(), f'Evre 1 kapısı için model bulunamadı: {checkpoint}'
    cascade_script = EXPERIMENT_DIR / 'scripts/cascade_eval.py'
    subprocess.run([
        sys.executable, '-u', str(cascade_script),
        '--data-root', str(DATA_ROOT), '--output-dir', str(OUTPUT_DIR),
        '--gate-checkpoints-dir', str(GATE_CHECKPOINTS_DIR),
        '--folds', FOLDS, '--batch', str(BATCH_SIZE), '--workers', str(WORKERS),
    ], check=True, cwd=EXPERIMENT_DIR)
    for report_path in sorted(RESULTS_DIR.glob('cascade*.csv')):
        print('\n' + report_path.name)
        display(pd.read_csv(report_path))
else:
    print('İsteğe bağlı tüm-sistem değerlendirmesi kapalı. Üç sınıflı ana sonuçlar hazır.')

## 10. Sonuçları ZIP olarak indir

Tablolar, görüntü bazında test tahminleri, fold listeleri ve deney ayarları tek ZIP'e konur. Büyük checkpoint dosyaları ZIP'e eklenmez; Drive'daki `checkpoints/` klasöründe kalır. ZIP'in bir kopyası da Drive'a kaydedilir. Otomatik indirme başlamazsa son bağlantıyı kullanın.

In [ ]:
from datetime import datetime, timezone
import shutil
from google.colab import files
from IPython.display import FileLink

assert (RESULTS_DIR / 'results.csv').is_file(), 'Önce eğitim/sonuç hücresini çalıştırın.'
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
zip_output = Path('/content') / f'exp15_stage234_tissue_results_{stamp}.zip'

# Only report files are packed; checkpoints and past exports stay out of this ZIP.
report_files = [path for path in RESULTS_DIR.rglob('*') if path.is_file()]
report_files.extend(path for path in OUTPUT_DIR.glob('*.json') if path.is_file())
with zipfile.ZipFile(zip_output, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(set(report_files)):
        if path.suffix not in {'.pt', '.pth', '.tmp'}:
            archive.write(path, arcname=path.relative_to(OUTPUT_DIR))
export_dir = OUTPUT_DIR / 'exports'
export_dir.mkdir(parents=True, exist_ok=True)
drive_copy = export_dir / zip_output.name
shutil.copy2(zip_output, drive_copy)
print('Sonuç ZIP:', zip_output)
print('Drive kopyası:', drive_copy)
print('Model dosyaları:', OUTPUT_DIR / 'checkpoints')
display(FileLink(str(zip_output)))
files.download(str(zip_output))